# PDF → Leitura em voz alta (TTS) com **pausas controladas** (Jupyter/Colab)

Este notebook faz:
1. Upload do PDF (Colab ou Jupyter com widget)
2. Extração de texto (PyMuPDF/pypdf)
3. (Opcional) OCR para PDF escaneado
4. Limpeza + normalização (evita leitura estranha de símbolos)
5. Segmentação por pontuação **(, ; : . ? !)**
6. Geração de áudio com `edge-tts` (sem SSML custom)
7. Merge final com **silêncio em ms por pontuação** (controle total das pausas)

> Dica: ajuste `PAUSE_MS` e `RATE` para ficar com “cara de audiolivro”.

In [ ]:
# ===== 1) Instalação de dependências =====
# Rode esta célula uma vez.
!pip -q install pymupdf pypdf edge-tts nest_asyncio ipywidgets langdetect

# Opcional (TTS local com GPU): descomente para instalar Coqui TTS.
# Compatibilidade no Colab (Python 3.x):
# - pip: !pip -q install TTS==0.22.0 (ou outra versão compatível com seu Python)
# - sem wheel: !pip -q install git+https://github.com/coqui-ai/TTS
# !pip -q install TTS==0.22.0
# !pip -q install git+https://github.com/coqui-ai/TTS
# Após a instalação bem-sucedida, ative o backend local definindo TTS_BACKEND="coqui".

# Para MERGE em um único MP3, usaremos pydub + ffmpeg.
# No Colab, a célula abaixo instala ffmpeg automaticamente.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.8 MB/s eta 0:00:00


In [ ]:
# ===== 2) (Recomendado) Instalar ffmpeg para juntar os áudios =====
# - Colab: instala via apt-get
# - Jupyter local: instale ffmpeg no seu sistema (Windows/Linux/Mac) para:
#   - MERGE (pydub)
#   - Pós-processamento (normalização LUFS, compressão leve e limiter)
#
# Parâmetros recomendados (ajuste se quiser):
# - Loudness: -16 LUFS (podcast) ou -18 LUFS (audiobook)
# - True Peak: -1.5 dB
# - LRA: 11
# - Compressor: threshold -18 dB, ratio 2:1
# - Limiter: -1.0 dB

import sys

def running_in_colab():
    return "google.colab" in sys.modules

if running_in_colab():
    !apt-get -y update >/dev/null
    !apt-get -y install ffmpeg >/dev/null
    print("ffmpeg instalado (Colab).")
else:
    print("Não-Colab: se o merge/pós-processamento falhar, instale ffmpeg no seu sistema.")

!pip -q install pydub


In [ ]:
# ===== 3) Upload do PDF (Colab ou Jupyter) =====
from pathlib import Path
import sys

PDF_PATH = None

def running_in_colab():
    return "google.colab" in sys.modules

if running_in_colab():
    from google.colab import files
    up = files.upload()
    if not up:
        raise RuntimeError("Nenhum arquivo foi enviado.")
    fname = next(iter(up.keys()))
    PDF_PATH = str(Path(fname).resolve())
    print("Arquivo carregado:", PDF_PATH)
else:
    # Jupyter local: tenta ipywidgets FileUpload
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output

        uploader = widgets.FileUpload(accept=".pdf", multiple=False)
        display(uploader)
        print("↑ Selecione um PDF acima e RE-EXECUTE esta célula para salvar o arquivo.")

        if uploader.value:
            item = next(iter(uploader.value.values()))
            content = item.get("content", None)
            name = item.get("metadata", {}).get("name", "arquivo.pdf")

            if content is None:
                content = item
                name = getattr(item, "name", "arquivo.pdf")

            out = Path(name).name
            Path(out).write_bytes(content)
            PDF_PATH = str(Path(out).resolve())
            clear_output()
            print("Arquivo carregado:", PDF_PATH)
    except Exception as e:
        print("Não consegui ativar upload por widget aqui.")
        print("Alternativas:")
        print("1) Coloque o PDF na mesma pasta do notebook e defina PDF_PATH manualmente.")
        print("Erro:", repr(e))

# Se precisar setar manualmente, descomente e ajuste:
# PDF_PATH = r"caminho/para/seu_arquivo.pdf"

In [ ]:
# ===== 4) Extrair texto do PDF =====
import re
import pymupdf  # PyMuPDF
from pypdf import PdfReader

def extract_text_pymupdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    parts = []
    for page in doc:
        parts.append(page.get_text("text") or "")
    return "\n".join(parts)

def extract_text_pypdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or "")
    return "\n".join(parts)

def extract_text_best_effort(pdf_path: str) -> str:
    t1 = ""
    try:
        t1 = extract_text_pymupdf(pdf_path)
    except Exception as e:
        print("PyMuPDF falhou, tentando pypdf. Erro:", repr(e))

    t2 = ""
    try:
        t2 = extract_text_pypdf(pdf_path)
    except Exception as e:
        print("pypdf falhou. Erro:", repr(e))

    return t1 if len(t1) >= len(t2) else t2

if not PDF_PATH:
    raise RuntimeError("PDF_PATH está vazio. Volte na célula de upload e carregue um PDF.")

raw_text = extract_text_best_effort(PDF_PATH)
print("Caracteres extraídos:", len(raw_text))
print("\nAmostra:\n", raw_text[:1200])

In [ ]:
# ===== 5) (Opcional) OCR se o PDF for escaneado =====
# Se quase não saiu texto (ex.: < 500 caracteres), ative USE_OCR=True.
# OBS: OCR pode exigir instalação de Tesseract e Poppler no sistema.
#
# Colab (recomendado):
#   !apt-get -y update
#   !apt-get -y install -y tesseract-ocr poppler-utils
#   !pip -q install pdf2image pytesseract
#
USE_OCR = False

def needs_ocr(text: str, threshold: int = 500) -> bool:
    return (text is None) or (len(text.strip()) < threshold)

if USE_OCR or needs_ocr(raw_text):
    print("Texto parece insuficiente. Tentando OCR...")
    try:
        !pip -q install pdf2image pytesseract
        if running_in_colab():
            !apt-get -y install -y tesseract-ocr poppler-utils >/dev/null
        from pdf2image import convert_from_path
        import pytesseract

        def ocr_pdf(pdf_path: str, dpi: int = 200, max_pages=None) -> str:
            images = convert_from_path(pdf_path, dpi=dpi)
            if max_pages is not None:
                images = images[:max_pages]
            parts = []
            for i, img in enumerate(images, 1):
                txt = pytesseract.image_to_string(img, lang="por")
                parts.append(txt)
                print(f"OCR página {i}/{len(images)} OK (chars: {len(txt)})")
            return "\n".join(parts)

        # Dica: para testar rápido, limite páginas:
        # raw_text = ocr_pdf(PDF_PATH, dpi=200, max_pages=5)
        raw_text = ocr_pdf(PDF_PATH, dpi=200, max_pages=None)
        print("OCR concluído. Caracteres:", len(raw_text))
    except Exception as e:
        print("Falha no OCR. Prováveis causas: falta Tesseract/Poppler no sistema.")
        print("Erro:", repr(e))
else:
    print("OCR não necessário (ou USE_OCR=False).")

In [ ]:
# ===== 6) Limpeza + normalização para TTS =====
# Objetivo: melhorar fluidez, manter parágrafos e evitar pronúncia “estranha” de símbolos.

PARA_TOKEN = "<PARA>"

LANGUAGE_FEATURE_RULES = {
    "date": {
        "pattern": r"(\d{2})/(\d{2})/(\d{4})",
        "repl": lambda m: f"dia {int(m.group(1))} de {int(m.group(2))} de {m.group(3)}",
    },
    "large_numbers": {
        "pattern": r"\d{1,3}(?:\.\d{3})+",
        "repl": lambda m: m.group(0).replace(".", " "),
    },
    "ordinals": {
        "pattern": r"(\d+)(º|ª)",
        "repl": lambda m: f"{m.group(1)} {m.group(2)}",
    },
    "acronyms": {
        "pattern": r"[A-Z]{2,}",
        "repl": lambda m: " ".join(m.group(0)),
    },
    "units": {
        "pattern": r"(\d+)\s?(km|kg)",
        "repl": lambda m: f"{m.group(1)} {'quilômetros' if m.group(2) == 'km' else 'quilos'}",
    },
    "percent": {
        "pattern": r"(\d+)\s?%",
        "repl": lambda m: f"{m.group(1)} por cento",
    },
}

def clean_text_basic(t: str) -> str:
    t = (t or "").replace(" ", " ")
    # remove hifenização por quebra de linha: "exem-
plo" -> "exemplo"
    t = re.sub(r"(\w)-
(\w)", r"", t)
    # normaliza quebras e espaços
    t = re.sub(r"
", "
", t)
    # preserva parágrafos (dupla quebra) para pausas maiores
    t = re.sub(r"
{2,}", "

", t)
    t = re.sub(r"[ 	]+", " ", t)
    # quebra simples vira espaço quando não é final de frase
    t = re.sub(r"(?<![.!?…])
", " ", t)
    t = re.sub(r" {2,}", " ", t)
    return t.strip()

def normalize_for_tts(t: str) -> str:
    # bullets e travessões viram pausa de frase
    t = t.replace("•", ". ").replace("–", ". ").replace("—", ". ")
    # listas tipo " - item" viram frase (evita falar "hífen")
    t = re.sub(r"\s-\s", ". ", t)
    # símbolos comuns que costumam soar estranhos
    t = t.replace("§", " seção ")
    t = t.replace("%", " por cento ")
    # preserva parágrafos para pausa extra
    t = t.replace("

", f" {PARA_TOKEN} ")
    # normaliza pontos repetidos
    t = re.sub(r"[.]{2,}", ".", t)
    t = re.sub(r"\s{2,}", " ", t)
    return t.strip()

def normalize_language_features(t: str, rules: dict | None = None) -> str:
    rules = rules or LANGUAGE_FEATURE_RULES
    for rule in rules.values():
        t = re.sub(rule["pattern"], rule["repl"], t)
    return t

text_clean = normalize_language_features(normalize_for_tts(clean_text_basic(raw_text)))
print("Caracteres após limpeza:", len(text_clean))
print("
Amostra:
", text_clean[:1200])

# logs para inspecionar mudanças
sample_phrases = [
    "Evento em 05/08/2024 com 12.500 pessoas da ONU.",
    "Peso: 3kg, distância 10km, desconto 15%.",
    "Ele ficou em 2º lugar na etapa 1ª.",
]
for phrase in sample_phrases:
    normalized = normalize_language_features(normalize_for_tts(phrase))
    print("
Original:", phrase)
    print("Normalizado:", normalized)


In [ ]:
# ===== 7) Segmentação por pontuação (controle fino de pausas) =====
# Vamos quebrar o texto em segmentos que terminam em: , ; : . ? !
# Depois, no merge, inserimos silêncio baseado nessa pontuação.
# Também preservamos parágrafos para pausas maiores (audiobook).

# Ajuste de tamanho dos segmentos (audiobook tende a ficar mais natural em blocos médios)
TARGET_MIN_CHARS = 160
TARGET_MAX_CHARS = 360


from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

LANG_MAP = {
    "pt": "pt-BR",
    "en": "en-US",
    "es": "es-ES",
}

FORCE_LANG = None  # Ex.: "pt-BR" para forçar idioma em todos os segmentos

def split_with_punct(text: str):
    text = (text or "").strip()
    if not text:
        return []

    # garante espaço depois de pontuação (ajuda o split)
    text = re.sub(r"([,;:\.\?!])(?=\S)", r"\1 ", text)
    text = text.replace(PARA_TOKEN, f" {PARA_TOKEN} ")

    paragraphs = [p.strip() for p in text.split(PARA_TOKEN)]

    out = []
    for idx, para in enumerate(paragraphs):
        if not para:
            continue
        parts = re.findall(r".+?[\,\.;:\?!](?:\s+|$)|.+$", para, flags=re.DOTALL)
        for p in parts:
            p = p.strip()
            if not p:
                continue
            punct = p[-1] if p[-1] in ",;:.?!" else "."
            out.append((p, punct, False))
        # marca quebra de parágrafo no último segmento do parágrafo
        if idx < len(paragraphs) - 1 and out:
            last = out[-1]
            out[-1] = (last[0], last[1], True)
    return out


def merge_short_segments(segments, min_chars=TARGET_MIN_CHARS, max_chars=TARGET_MAX_CHARS):
    merged = []
    buffer = ""
    buffer_punct = "."
    buffer_para = False

    for txt, punct, para in segments:
        if not buffer:
            buffer = txt
            buffer_punct = punct
            buffer_para = para
        else:
            candidate = f"{buffer} {txt}".strip()
            if len(candidate) <= max_chars:
                buffer = candidate
                buffer_punct = punct
                buffer_para = para
            else:
                merged.append((buffer, buffer_punct, buffer_para))
                buffer = txt
                buffer_punct = punct
                buffer_para = para

        if len(buffer) >= min_chars and buffer_para:
            merged.append((buffer, buffer_punct, buffer_para))
            buffer = ""
            buffer_punct = "."
            buffer_para = False

    if buffer:
        merged.append((buffer, buffer_punct, buffer_para))

    return merged



def detect_lang(text: str) -> str:
    if FORCE_LANG:
        return FORCE_LANG
    try:
        lang = detect(text)
    except Exception:
        return "pt-BR"
    return LANG_MAP.get(lang, "pt-BR")


def add_lang_to_segments(segments):
    return [(txt, punct, para_break, detect_lang(txt)) for txt, punct, para_break in segments]

segments = add_lang_to_segments(merge_short_segments(split_with_punct(text_clean)))

print("Total de segmentos:", len(segments))
print("Exemplo 1:", segments[0][0][:180], "| punct:", segments[0][1] if segments else None)
print("Exemplo 1 idioma:", segments[0][3] if segments else None)


In [ ]:
# ===== 8) Escolha de voz (robusta: não quebra se uma falhar) =====
import os, asyncio
import nest_asyncio
nest_asyncio.apply()
import edge_tts

from IPython.display import Audio, display

async def list_voices_by_locale(locale: str):
    voices = await edge_tts.list_voices()
    return sorted({v["ShortName"] for v in voices if v.get("Locale") == locale})

async def list_ptbr_voices():
    return await list_voices_by_locale("pt-BR")

def filter_multilingual(voices):
    return [v for v in voices if "Multilingual" not in v]

async def synth_one(ssml, voice, rate, pitch, out_path, retries=2):
    last_err = None
    for attempt in range(retries + 1):
        try:
            comm = edge_tts.Communicate(text=ssml, voice=voice, rate=rate, pitch=pitch)
            await comm.save(out_path)
            return True
        except Exception as e:
            last_err = e
            await asyncio.sleep(0.8 * (attempt + 1))
    print(f"Falhou {voice}: {type(last_err).__name__}: {last_err}")
    return False

ptbr = await list_ptbr_voices()
ptbr_non_multi = filter_multilingual(ptbr)

print("Vozes pt-BR disponíveis agora:", ptbr)
if ptbr_non_multi:
    print("Vozes pt-BR (sem Multilingual):", ptbr_non_multi)

SAMPLE_TEXT = (
    "Teste de voz. Pausas e naturalidade importam. "
    "Se estiver rápido demais, vou falar um pouco mais devagar, ok?"
)

SAMPLE_OUT = "samples_voz"
os.makedirs(SAMPLE_OUT, exist_ok=True)

# Ajuste aqui (mais lento costuma soar menos robótico)
TEST_RATE  = "-10%"
TEST_PITCH = "-2Hz"

# Testa até 5 vozes (preferindo não-multilingual)
VOICES_TO_TEST = (ptbr_non_multi or ptbr)[:5]

sample_files = []
for v in VOICES_TO_TEST:
    outp = os.path.join(SAMPLE_OUT, f"{v}.mp3")
    ok = await synth_one(SAMPLE_TEXT, v, TEST_RATE, TEST_PITCH, outp, retries=2)
    if ok:
        print("OK:", outp)
        sample_files.append(outp)

for f in sample_files:
    print(f)
    display(Audio(f, autoplay=False))


In [ ]:
# ===== 9) Configuração de vozes por idioma =====
PREFERRED_VOICES_BY_LANG = {
    "pt-BR": [
        "pt-BR-FranciscaNeural",
        "pt-BR-AntonioNeural",
        "pt-BR-ElzaNeural",
        "pt-BR-FabioNeural",
    ],
    "en-US": [
        "en-US-JennyNeural",
        "en-US-GuyNeural",
        "en-US-AriaNeural",
    ],
    "es-ES": [
        "es-ES-ElviraNeural",
        "es-ES-AlvaroNeural",
    ],
}

async def list_voices_map():
    return {
        lang: filter_multilingual(await list_voices_by_locale(lang))
        for lang in PREFERRED_VOICES_BY_LANG
    }

def choose_voice_for_lang(lang: str, available_map, fallback="pt-BR"):
    available = available_map.get(lang) or []
    preferred = PREFERRED_VOICES_BY_LANG.get(lang) or []
    for v in preferred:
        if v in available:
            return v
    if available:
        return available[0]
    if lang != fallback:
        return choose_voice_for_lang(fallback, available_map, fallback=fallback)
    return PREFERRED_VOICES_BY_LANG[fallback][0]

voice_map = await list_voices_map()


In [ ]:
# ===== 10) Gerar áudio por segmento (voz por idioma detectado) =====

# Preferência por vozes sem "Multilingual" reduz variação de sotaque.
RATE  = "-10%"  # mais lento = mais natural (audiobook)
PITCH = "-2Hz"
VOLUME = "+0dB"

# Backend de TTS: "edge" (online) ou "coqui" (local, usa GPU se disponível)
TTS_BACKEND = "edge"
MAX_CONCURRENCY = 5  # edge-tts é I/O bound; para TTS local use 1-2
COQUI_MODEL_NAME = "tts_models/pt/cv/vits"

# SSML (use apenas em segmentos grandes para evitar limites)
USE_SSML = True
SSML_MIN_CHARS = 220
SSML_PAUSE_MS = {
    ",": 240,
    ";": 480,
    ":": 480,
    ".": 520,
    "?": 560,
    "!": 560,
}
SSML_PARA_BREAK_MS = 750

OUT_DIR = "out_audio_segs"
os.makedirs(OUT_DIR, exist_ok=True)
MANIFEST_PATH = os.path.join(OUT_DIR, "segments_manifest.json")

# Para teste rápido, limite o número de segmentos (ex.: 200)
LIMIT = None  # ou 200

import asyncio
import html
import hashlib
import json
from datetime import datetime


_local_tts = None


def get_local_tts():
    global _local_tts
    if _local_tts is None:
        from TTS.api import TTS
        import torch

        device = "cuda" if torch.cuda.is_available() else "cpu"
        _local_tts = TTS(COQUI_MODEL_NAME).to(device)
    return _local_tts


def synth_local(text: str, out_path: str) -> None:
    tts = get_local_tts()
    tts.tts_to_file(text=text, file_path=out_path)


def build_ssml(text: str, punct: str, para_break: bool) -> str:
    safe_text = html.escape(text or "", quote=False)
    pause = SSML_PAUSE_MS.get(punct, 320)
    if para_break:
        pause += SSML_PARA_BREAK_MS
    pause_tag = f'<break time="{pause}ms"/>' if pause else ""
    return (
        f"<speak>"
        f"<prosody rate="{RATE}" pitch="{PITCH}" volume="{VOLUME}">"
        f"{safe_text}"
        f"</prosody>"
        f"{pause_tag}"
        f"</speak>"
    )


def segment_hash(text: str, voice: str, rate: str, pitch: str, backend: str, model: str) -> str:
    payload = f"{backend}|{model}|{text}|{voice}|{rate}|{pitch}"
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()


def load_manifest(path: str = MANIFEST_PATH) -> dict | None:
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def save_manifest(manifest: dict, path: str = MANIFEST_PATH) -> None:
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)


async def synthesize_one_segment(index, txt, punct, para_break, lang, rate, pitch, out_dir):
    voice = choose_voice_for_lang(lang, voice_map)
    model_name = COQUI_MODEL_NAME if TTS_BACKEND == "coqui" else "edge"
    seg_hash = segment_hash(txt, voice, rate, pitch, TTS_BACKEND, model_name)
    filename = f"seg_{seg_hash}.mp3"
    out_path = os.path.join(out_dir, filename)
    use_ssml = USE_SSML and len(txt) >= SSML_MIN_CHARS and TTS_BACKEND == "edge"
    ssml = build_ssml(txt, punct, para_break) if use_ssml else txt
    existed = os.path.exists(out_path)
    ok = True
    synthesized = False
    if not existed:
        if TTS_BACKEND == "edge":
            ok = await synth_one(ssml, voice, rate, pitch, out_path, retries=2)
            synthesized = ok
        else:
            await asyncio.to_thread(synth_local, txt, out_path)
            synthesized = True
    if not ok:
        out_path = None
    return {
        "index": index,
        "hash": seg_hash,
        "file": out_path,
        "punct": punct,
        "para_break": para_break,
        "lang": lang,
        "voice": voice,
        "rate": rate,
        "pitch": pitch,
        "text": txt,
        "ssml": use_ssml,
        "existed": existed,
        "synthesized": synthesized,
        "backend": TTS_BACKEND,
        "model": model_name,
    }


async def synthesize_segments(segments, out_dir=OUT_DIR, rate=RATE, pitch=PITCH, limit=None):
    todo = segments if limit is None else segments[:limit]
    manifest = {
        "created_at": datetime.utcnow().isoformat() + "Z",
        "out_dir": out_dir,
        "rate": rate,
        "pitch": pitch,
        "volume": VOLUME,
        "backend": TTS_BACKEND,
        "model": COQUI_MODEL_NAME if TTS_BACKEND == "coqui" else "edge",
        "segments": [],
    }
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)

    async def run_one(index, txt, punct, para_break, lang):
        async with semaphore:
            return await synthesize_one_segment(
                index,
                txt,
                punct,
                para_break,
                lang,
                rate,
                pitch,
                out_dir,
            )

    tasks = [
        asyncio.create_task(run_one(i, txt, punct, para_break, lang))
        for i, (txt, punct, para_break, lang) in enumerate(todo, 1)
    ]

    completed = 0
    for task in asyncio.as_completed(tasks):
        entry = await task
        manifest["segments"].append(entry)
        completed += 1
        if entry["lang"] != "pt-BR":
            print(f"[lang={entry['lang']}] seg {entry['index']}: {entry['text'][:80]}...")
        if completed % 50 == 0:
            print(f"{completed}/{len(todo)}")

    manifest["segments"].sort(key=lambda item: item["index"])
    save_manifest(manifest)
    return manifest

seg_manifest = await synthesize_segments(segments, limit=LIMIT)
print("Segmentos no manifest:", len(seg_manifest["segments"]))


## Exemplos de SSML para diálogo com vozes diferentes

Use `voice name="..."` para alternar personagens no mesmo trecho.

```xml
<speak>
  <voice name="pt-BR-AntonioNeural">
    — Você viu o livro?
  </voice>
  <break time="250ms"/>
  <voice name="pt-BR-FranciscaNeural">
    — Ainda não, mas vou procurar.
  </voice>
</speak>
```

Outro exemplo combinando prosódia:

```xml
<speak>
  <voice name="pt-BR-AntonioNeural">
    <prosody rate="-5%" pitch="-2Hz">Que noite longa...</prosody>
  </voice>
  <voice name="pt-BR-FranciscaNeural">
    <prosody rate="+0%" pitch="+1Hz">Concordo, mas ainda temos trabalho.</prosody>
  </voice>
</speak>
```


In [ ]:
# ===== 11) Merge final com pausas configuráveis por pontuação =====
from pydub import AudioSegment
from IPython.display import Audio, display
import os
import shutil
import subprocess
import json

# Ajuste as pausas aqui (em milissegundos)
PAUSE_MS = {
    ",": 240,
    ";": 480,
    ":": 480,
    ".": 520,
    "?": 560,
    "!": 560,
}

# Pausa adicional para quebra de parágrafo (audiobook)
PARA_BREAK_MS = 750

MERGE = True
POST_PROCESS = True

# Configurações recomendadas para pós-processamento
TARGET_LUFS = -16  # ou -18 para audiobook
TRUE_PEAK_DB = -1.5
LRA = 11
COMP_THRESHOLD_DB = -18
COMP_RATIO = 2
LIMIT_DB = -1.0


def postprocess_audio(input_path, output_path):
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg não encontrado. Instale-o para usar o pós-processamento.")

    ffmpeg_cmd = [
        "ffmpeg",
        "-y",
        "-i",
        input_path,
        "-filter:a",
        (
            f"loudnorm=I={TARGET_LUFS}:TP={TRUE_PEAK_DB}:LRA={LRA},"
            f"acompressor=threshold={COMP_THRESHOLD_DB}dB:ratio={COMP_RATIO},"
            f"alimiter=limit={LIMIT_DB}dB"
        ),
        "-codec:a",
        "libmp3lame",
        "-q:a",
        "2",
        output_path,
    ]
    subprocess.run(ffmpeg_cmd, check=True)


def load_manifest(path: str = MANIFEST_PATH) -> dict | None:
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


if MERGE:
    combined = AudioSegment.empty()
    count_added = 0

    manifest = load_manifest()
    if not manifest:
        raise RuntimeError("Manifest não encontrado. Rode a etapa de síntese primeiro.")

    for entry in manifest.get("segments", []):
        path = entry.get("file")
        punct = entry.get("punct", ".")
        para_break = entry.get("para_break", False)
        if path and os.path.exists(path):
            combined += AudioSegment.from_file(path, format="mp3")
            pause = PAUSE_MS.get(punct, 320)
            if para_break:
                pause += PARA_BREAK_MS
            combined += AudioSegment.silent(duration=pause)
            count_added += 1

    raw_path = "audio_final.mp3"
    combined.export(raw_path, format="mp3")

    final_path = raw_path
    if POST_PROCESS:
        final_path = "audio_final_post.mp3"
        postprocess_audio(raw_path, final_path)

    print("Segmentos adicionados:", count_added)
    print("Arquivo final:", final_path)
    display(Audio(final_path, autoplay=False))
else:
    print("MERGE=False. Os segmentos estão em:", OUT_DIR)


## Ajustes rápidos (como “tunar” o resultado)

- **Pausas:** edite `PAUSE_MS` (ms) na célula de merge.
  - Quer mais pausa na vírgula? aumente `","` para 300–450.
  - Quer pausa “dramática” em ponto final? aumente `"."` para 800–1100.

- **Naturalidade:** ajuste `RATE` e o mapa `PREFERRED_VOICES_BY_LANG`.
  - `RATE=-8%` ou `-10%` geralmente fica menos robótico.
  - Troque a ordem das vozes por idioma na célula de configuração.


- **Forçar idioma manualmente:** defina `FORCE_LANG = "pt-BR"` (ou `"en-US"`, `"es-ES"`) na célula de segmentação antes de gerar os segmentos.

- **PDF escaneado:** ative `USE_OCR=True` (a célula de OCR explica dependências).

- **Teste rápido:** use `LIMIT=200` para não demorar num PDF grande.